# Lab 4: Grid Sorting — SOM and FLAS

**Objectives:**
- Implement D_p path metric and Grid Adjacency Fidelity (GAF)
- Build a Self-Organizing Map (SOM) for 2D grid sorting
- Implement simplified FLAS (filter + local linear assignment)
- Compare quality and speed on RGB color sorting
- Demonstrate compression benefit via entropy reduction after sorting


In [ ]:
!pip install -q numpy matplotlib scipy scikit-learn
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from scipy.stats import entropy as scipy_entropy
import time
%matplotlib inline
print('Ready.')

## Section 1: Quality Metrics — D_p and GAF

In [ ]:
def dp_metric(arr, p=2):
    """D_p path metric: mean p-norm distance between consecutive elements."""
    diffs = np.diff(arr, axis=0)
    dists = np.linalg.norm(diffs, axis=-1) if arr.ndim > 1 else np.abs(diffs)
    return float((np.mean(dists**p))**(1/p))

def grid_adjacency_fidelity(grid):
    """Mean Euclidean distance to 4-connected grid neighbors (lower = better)."""
    dists = []
    dists.append(np.linalg.norm(grid[:, :-1] - grid[:, 1:],  axis=-1).flatten())
    dists.append(np.linalg.norm(grid[:-1, :] - grid[1:, :],  axis=-1).flatten())
    return float(np.mean(np.concatenate(dists)))

# Demonstrate on 1D data
data = np.array([3.1, 1.4, 5.9, 2.6, 5.3, 8.9, 7.9, 3.2])
sorted_data = np.sort(data)
random_data = np.random.permutation(data)
print(f'Original D_p: {dp_metric(data[:,None]):.3f}')
print(f'Sorted D_p:   {dp_metric(sorted_data[:,None]):.3f}  <- optimal')
print(f'Random D_p:   {dp_metric(random_data[:,None]):.3f}')

# Show sorting quality on random 2D arrangement of RGB colors
np.random.seed(42)
N = 1024
colors = np.random.rand(N, 3)
grid_H, grid_W = 32, 32
random_grid = colors.reshape(grid_H, grid_W, 3)
print(f'\nRandom grid GAF: {grid_adjacency_fidelity(random_grid):.4f}')

## Section 2: Self-Organizing Map (SOM)

In [ ]:
def som_sort(data, H, W, n_iters=8000, lr0=0.5, radius0=None):
    """
    Sort N D-dimensional vectors onto H×W grid using SOM.
    Returns (H, W, D) grid of prototype vectors.
    """
    N, D = data.shape
    if radius0 is None:
        radius0 = max(H, W) / 2.0
    np.random.seed(0)
    grid = (data[np.random.choice(N, H*W, replace=False)]
            .reshape(H, W, D).copy())
    gy, gx = np.mgrid[0:H, 0:W]  # (H, W)
    for t in range(n_iters):
        frac = t / n_iters
        lr = lr0 * np.exp(-frac * 4)
        rad = radius0 * np.exp(-frac * 4) + 0.5
        x = data[np.random.randint(N)]
        dists = np.linalg.norm(grid - x, axis=-1)
        by, bx = np.unravel_index(np.argmin(dists), (H, W))
        h = np.exp(-((gy-by)**2 + (gx-bx)**2) / (2*rad**2))[:, :, None]
        grid += lr * h * (x - grid)
    return grid

print('Running SOM (N=1024 colors onto 32×32 grid)...')
t0 = time.time()
som_grid = som_sort(colors, grid_H, grid_W, n_iters=10000)
print(f'SOM done in {time.time()-t0:.1f}s')

# Assign each color to its nearest SOM prototype
grid_flat = som_grid.reshape(-1, 3)
assignments = np.argmin(cdist(colors, grid_flat), axis=1)
assigned = np.zeros((grid_H, grid_W, 3))
for i, c in enumerate(colors):
    r, col = np.unravel_index(assignments[i], (grid_H, grid_W))
    assigned[r, col] = c

gaf_random = grid_adjacency_fidelity(random_grid)
gaf_som = grid_adjacency_fidelity(assigned)
print(f'\nGAF random: {gaf_random:.4f}')
print(f'GAF SOM:    {gaf_som:.4f}  ({gaf_random/gaf_som:.2f}x improvement)')

## Section 3: Fast Linear Assignment Sorting (FLAS)

In [ ]:
def flas_sort(data, H, W, n_iters=8, sigma=3.0, patch=8):
    """
    Simplified FLAS: Gaussian filter → local linear assignment, repeated.
    data: (N, D) where N = H * W
    Returns: (H, W, D) sorted grid
    """
    assert data.shape[0] == H * W
    N, D = data.shape
    np.random.seed(1)
    grid = data[np.random.permutation(N)].reshape(H, W, D).copy()

    for iteration in range(n_iters):
        # Step 1: Gaussian filter (global smoothing for context)
        smoothed = np.stack([
            gaussian_filter(grid[:,:,d], sigma=sigma) for d in range(D)
        ], axis=-1)

        # Step 2: Local linear assignment in patches
        for i in range(0, H - patch + 1, patch // 2):
            for j in range(0, W - patch + 1, patch // 2):
                ie, je = min(i+patch, H), min(j+patch, W)
                p_data   = grid[i:ie, j:je].reshape(-1, D)
                p_target = smoothed[i:ie, j:je].reshape(-1, D)
                if len(p_data) < 2:
                    continue
                cost = cdist(p_data, p_target)
                row_idx, col_idx = linear_sum_assignment(cost)
                reordered = p_data[row_idx][np.argsort(col_idx)]
                grid[i:ie, j:je] = reordered.reshape(ie-i, je-j, D)
    return grid

print('Running FLAS...')
t0 = time.time()
flas_grid = flas_sort(colors, grid_H, grid_W, n_iters=6, sigma=3.0, patch=8)
flas_time = time.time() - t0
gaf_flas = grid_adjacency_fidelity(flas_grid)
print(f'FLAS done in {flas_time:.2f}s')
print(f'GAF FLAS:   {gaf_flas:.4f}  ({gaf_random/gaf_flas:.2f}x improvement)')

print(f'\nMethod    | GAF    | Time  | Improvement')
print(f'----------|--------|-------|------------')
print(f'Random    | {gaf_random:.4f} | -     | 1.00x')
print(f'SOM       | {gaf_som:.4f} | ~10s  | {gaf_random/gaf_som:.2f}x')
print(f'FLAS      | {gaf_flas:.4f} | {flas_time:.1f}s  | {gaf_random/gaf_flas:.2f}x')

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(random_grid); axes[0].set_title(f'Random\nGAF={gaf_random:.4f}'); axes[0].axis('off')
axes[1].imshow(assigned);    axes[1].set_title(f'SOM\nGAF={gaf_som:.4f}'); axes[1].axis('off')
axes[2].imshow(flas_grid);   axes[2].set_title(f'FLAS\nGAF={gaf_flas:.4f}'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## Section 4: Compression Benefit via Entropy Reduction

After sorting, adjacent Gaussians are more similar → smaller deltas → lower entropy → better compression.

In [ ]:
def array_entropy_bits(arr, n_bins=64):
    hist, _ = np.histogram(arr.flatten(), bins=n_bins)
    hist = hist[hist > 0]
    return float(scipy_entropy(hist / hist.sum(), base=2))

def delta_entropy(grid_2d_channel):
    """Entropy of horizontal differences (prediction residuals)."""
    deltas = np.diff(grid_2d_channel.flatten())
    return array_entropy_bits(deltas)

print('Entropy of horizontal deltas (R channel):')
print(f'  Random grid: {delta_entropy(random_grid[:,:,0]):.3f} bits')
print(f'  SOM grid:    {delta_entropy(assigned[:,:,0]):.3f} bits')
print(f'  FLAS grid:   {delta_entropy(flas_grid[:,:,0]):.3f} bits')

print('\nEntropy of horizontal deltas (G channel):')
print(f'  Random grid: {delta_entropy(random_grid[:,:,1]):.3f} bits')
print(f'  FLAS grid:   {delta_entropy(flas_grid[:,:,1]):.3f} bits')

print('\n=> Sorted grids have much lower delta entropy = better arithmetic coding compression')

# Visualize horizontal differences
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, grid, title in zip(axes,
    [random_grid, assigned, flas_grid],
    ['Random', 'SOM', 'FLAS']):
    diff_img = np.abs(np.diff(grid[:,:,0], axis=1))
    im = ax.imshow(diff_img, cmap='hot', vmin=0, vmax=0.3)
    ax.set_title(f'{title}\nHoriz. diff magnitude'); ax.axis('off')
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

## Key Takeaways

- **D_p metric** measures path length through sorted data — optimal sorting minimizes it
- **GAF** (Grid Adjacency Fidelity) measures mean neighbor distance — lower = better 2D sorting
- **SOM** sorts through sequential neighborhood pulls — good quality but creates outliers in dense grids
- **FLAS** (filter + local assignment) achieves better GAF than SOM, often in less time
- Sorted grids have **dramatically lower horizontal-delta entropy** → 3–9× better compressibility under image codecs
- FLAS scales to millions of Gaussians; LAS (global assignment) does not — O(N³) is the key bottleneck
